## Vector Stores and Retrivers

Vector Stores and Retrievers are the two core components that allow an AI to "remember" or access specific information that wasn't included in its original training data.

1. Vector Stores (The Bookshelf)
Traditional databases search for exact matches (like searching for the word "apple"). A Vector Store, however, searches by meaning.

How it Works:
Embeddings: Text is converted into a long list of numbers called an embedding. These numbers represent the semantic meaning of the text.

Multidimensional Space: These embeddings are placed in a geometric space. Concepts that are similar (e.g., "king" and "queen") are placed close together, while unrelated concepts (e.g., "king" and "potato") are far apart.

Storage: The Vector Store holds these numerical representations and their associated metadata (the original text).

Popular Examples:
Pinecone

ChromaDB

Weaviate

Milvus

2. Retrievers (The Librarian)
A Retriever is the interface that pulls the most relevant information out of the Vector Store when you ask a question.

The Process:
Query Conversion: When you ask a question, the Retriever converts your question into an embedding (numbers).

Similarity Search: It looks into the Vector Store to find the "nearest neighbors"—the pieces of text whose numbers most closely match the numbers of your question.

Output: It returns the top results (the "context") to the LLM so the AI can generate an answer based on those specific facts.

### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:

- page_content: a string representing the content;
- metadata: a dict containing arbitrary metadata.
The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger document.

Let's generate some sample documents:

In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")

os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

llm=ChatGroq(groq_api_key=groq_api_key,model="llama-3.1-8b-instant")
llm

d:\shree\project_files\samples\langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002221FB0A490>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002221FBCC790>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4223.00it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
## VectorStores
from langchain_chroma import Chroma

vectorstore=Chroma.from_documents(documents,embedding=embeddings)
vectorstore


In [6]:
vectorstore.similarity_search("cat")

[Document(id='8a973dfe-fb31-4eed-983e-0d645dc0c148', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='5e3c9b93-ced0-4340-a5bb-8c154261848c', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='c433dadf-7a75-4c8b-b82b-10c4865bf01e', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='4796fe70-5f1c-4a4e-bca8-092df9eb2bb6', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [7]:
## Async query
await vectorstore.asimilarity_search("cat")

[Document(id='8a973dfe-fb31-4eed-983e-0d645dc0c148', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='5e3c9b93-ced0-4340-a5bb-8c154261848c', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='c433dadf-7a75-4c8b-b82b-10c4865bf01e', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='4796fe70-5f1c-4a4e-bca8-092df9eb2bb6', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [9]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='8a973dfe-fb31-4eed-983e-0d645dc0c148', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351058006286621),
 (Document(id='5e3c9b93-ced0-4340-a5bb-8c154261848c', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740898847579956),
 (Document(id='c433dadf-7a75-4c8b-b82b-10c4865bf01e', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
  1.5956902503967285),
 (Document(id='4796fe70-5f1c-4a4e-bca8-092df9eb2bb6', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
  1.665792465209961)]

In [10]:
await vectorstore.asimilarity_search_with_score("cat")

[(Document(id='8a973dfe-fb31-4eed-983e-0d645dc0c148', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351058006286621),
 (Document(id='5e3c9b93-ced0-4340-a5bb-8c154261848c', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740898847579956),
 (Document(id='c433dadf-7a75-4c8b-b82b-10c4865bf01e', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
  1.5956902503967285),
 (Document(id='4796fe70-5f1c-4a4e-bca8-092df9eb2bb6', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
  1.665792465209961)]

### Retrievers
LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:

In [19]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever=RunnableLambda(vectorstore.similarity_search).bind(k=1) #The .bind() method is used to pre-configure a function with specific arguments before it is actually called.k=1: This tells the retriever to return only the one most relevant document
retriever.batch(["cat","dog"]) #The .batch() method allows you to process multiple inputs at once rather than one at a time.

[[Document(id='8a973dfe-fb31-4eed-983e-0d645dc0c148', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='5e3c9b93-ced0-4340-a5bb-8c154261848c', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [12]:
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cat","dog"])


[[Document(id='8a973dfe-fb31-4eed-983e-0d645dc0c148', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='5e3c9b93-ced0-4340-a5bb-8c154261848c', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [17]:
## RAG
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain={"context":retriever,"question":RunnablePassthrough()}|prompt|llm

response=rag_chain.invoke("tell me about birds")
print(response.content)


It appears that the context is a document about bird pets. Based on the provided information, here's what we know about birds:

Birds, specifically parrots, are intelligent animals. One notable characteristic of parrots is their ability to mimic human speech, making them interesting and engaging pets.
